In [ ]:
from numpy import ceil
from util import *

class RestrictedBoltzmannMachine():
    '''
    For more details : A Practical Guide to Training Restricted Boltzmann Machines https://www.cs.toronto.edu/~hinton/absps/guideTR.pdf
    '''
    def __init__(self, ndim_visible, ndim_hidden, is_bottom=False, image_size=[28,28], is_top=False, n_labels=10, batch_size=10):

        """
        Args:
          ndim_visible: Number of units in visible layer.
          ndim_hidden: Number of units in hidden layer.
          is_bottom: True only if this rbm is at the bottom of the stack in a deep belief net. Used to interpret visible layer as image data with dimensions "image_size".
          image_size: Image dimension for visible layer.
          is_top: True only if this rbm is at the top of stack in deep beleif net. Used to interpret visible layer as concatenated with "n_label" unit of label data at the end. 
          n_label: Number of label categories.
          batch_size: Size of mini-batch.
        """
       
        self.ndim_visible = ndim_visible

        self.ndim_hidden = ndim_hidden

        self.is_bottom = is_bottom

        if is_bottom : self.image_size = image_size
        
        self.is_top = is_top

        if is_top : self.n_labels = 10

        self.batch_size = batch_size        
                
        self.delta_bias_v = 0

        self.delta_weight_vh = 0

        self.delta_bias_h = 0

        self.bias_v = np.random.normal(loc=0.0, scale=0.01, size=(self.ndim_visible))

        self.weight_vh = np.random.normal(loc=0.0, scale=0.01, size=(self.ndim_visible,self.ndim_hidden))

        self.bias_h = np.random.normal(loc=0.0, scale=0.01, size=(self.ndim_hidden))
        
        self.delta_weight_v_to_h = 0

        self.delta_weight_h_to_v = 0        
        
        self.weight_v_to_h = None
        
        self.weight_h_to_v = None

        self.learning_rate = 0.01
        
        self.momentum = 0.7

        self.print_period = 5000
        
        self.rf = { # receptive-fields. Only applicable when visible layer is input data
            "period" : 5000, # iteration period to visualize
            "grid" : [5,5], # size of the grid
            "ids" : np.random.randint(0,self.ndim_hidden,25) # pick some random hidden units
            }
        
        return

        
    def cd1(self,visible_trainset, n_iterations=10000):
        
        """Contrastive Divergence with k=1 full alternating Gibbs sampling

        Args:
          visible_trainset: training data for this rbm, shape is (size of training set, size of visible layer)
          n_iterations: number of iterations of learning (each iteration learns a mini-batch)
        """

        print ("learning CD1")
        
        n_samples = visible_trainset.shape[0]



        batches_number = int(n_samples / self.batch_size)
        n_iterations = n_iterations * batches_number

        print(f"n_iterations={n_iterations} with batch_size={self.batch_size} and n_samples={n_samples}")

        w_change = []
        bv_change = []
        bh_change = []
        loss_history = []
        
        for it in range(n_iterations):
            start_index = (it * self.batch_size) % n_samples
            end_index = start_index + self.batch_size
            v_0 = visible_trainset[start_index:end_index, :]

        #for it in range(n_iterations):

	        # [TODO TASK 4.1] run k=1 alternating Gibbs sampling : v_0 -> h_0 ->  v_1 -> h_1.
            # you may need to use the inference functions 'get_h_given_v' and 'get_v_given_h'.
            # note that inference methods returns both probabilities and activations (samples from probablities) and you may have to decide when to use what.

            # visible_trainset has shape: (60000, 784)
            # self.weight_vh has shape: (784, 200)

            # (student)
            #for batch in range(batches_number):
            # We get the probabilities and activations of hidden layer
            # given the visibile training set, and use this to get
            # v_1 and h_1

            #start_index = batch * self.batch_size
            #end_index = min((batch + 1) * self.batch_size, n_samples)
            #v_0 = visible_trainset[start_index:end_index, :]

            p_h0, h0 = self.get_h_given_v(v_0)
            p_v1, v_1 = self.get_v_given_h(h0)
            p_h1, h1 = self.get_h_given_v(v_1)

            # [TODO TASK 4.1] update the parameters using function 'update_params'
            # (student)
            changes = self.update_params(v_0, h0, v_1, h1)
            # Only store history if batch is last in the epoch
            w_change.append(changes["w_change"])
            bv_change.append(changes["bv_change"]) 
            bh_change.append(changes["bh_change"])
            loss_history.append(np.linalg.norm(v_0 - v_1))
            
            # visualize once in a while when visible layer is input images

            #if it % self.rf["period"] == 0 and self.is_bottom:
            if it % batches_number == 0 and self.is_bottom:
                epoch = it // batches_number
                viz_rf(weights=self.weight_vh[:,self.rf["ids"]].reshape((self.image_size[0],self.image_size[1],-1)), it=epoch, grid=self.rf["grid"])

            # print progress
            
            #if it % self.print_period == 0 :
            if it % batches_number == 0 :
                epoch = it // batches_number
                print ("iteration=%7d recon_loss=%4.4f"%(epoch, loss_history[-1]))
                print(f"w_change={w_change[-1]:.6f} bv_change={bv_change[-1]:.6f} bh_change={bh_change[-1]:.6f}")
        
        return
    

    def update_params(self,v_0,h_0,v_k,h_k):

        """Update the weight and bias parameters.

        You could also add weight decay and momentum for weight updates.

        Args:
           v_0: activities or probabilities of visible layer (data to the rbm)
           h_0: activities or probabilities of hidden layer
           v_k: activities or probabilities of visible layer
           h_k: activities or probabilities of hidden layer
           all args have shape (size of mini-batch, size of respective layer)
        """

        # [TODO TASK 4.1] get the gradients from the arguments (replace the 0s below) and update the weight and bias parameters

        # (student)
        # Store the old weights and biases in order to compare change
        # with the goal of later displaying this data in order to track
        # the learning process
        if not hasattr(self, "old_weight_vh"):
            self.old_weight_vh = np.copy(self.weight_vh)
            self.old_bias_v = np.copy(self.bias_v)
            self.old_bias_h = np.copy(self.bias_h)

        # We impletement the update rules as seen in the slides,
        # but note the absense of the sum and normalization as this
        # is just a single update and not the full batch update.
        self.delta_bias_v = np.mean(v_0 - v_k, axis=0) * self.learning_rate + self.momentum * self.delta_bias_v
        self.delta_weight_vh = ((v_0.T @ h_0) - (v_k.T @ h_k)) / self.batch_size * self.learning_rate + self.momentum * self.delta_weight_vh
        self.delta_bias_h = np.mean(h_0 - h_k, axis=0) * self.learning_rate + self.momentum * self.delta_bias_h
        
        self.bias_v += self.delta_bias_v
        self.weight_vh += self.delta_weight_vh
        self.bias_h += self.delta_bias_h

        # Finally we calculate parameter changes for tracking purposes
        w_change = np.linalg.norm(self.old_weight_vh - self.weight_vh)
        bv_change = np.linalg.norm(self.old_bias_v - self.bias_v)
        bh_change = np.linalg.norm(self.old_bias_h - self.bias_h)
        # And update the old weights and biases for the next iteration
        self.old_weight_vh = np.copy(self.weight_vh)
        self.old_bias_v = np.copy(self.bias_v)
        self.old_bias_h = np.copy(self.bias_h)
        
        return {
            "w_change": w_change,
            "bv_change": bv_change,
            "bh_change": bh_change
        }

    def get_h_given_v(self,visible_minibatch):
        
        """Compute probabilities p(h|v) and activations h ~ p(h|v) 

        Uses undirected weight "weight_vh" and bias "bias_h"
        
        Args: 
           visible_minibatch: shape is (size of mini-batch, size of visible layer)
        Returns:        
           tuple ( p(h|v) , h) 
           both are shaped (size of mini-batch, size of hidden layer)
        """
        
        assert self.weight_vh is not None

        n_samples = visible_minibatch.shape[0]

        # [TODO TASK 4.1] compute probabilities and activations (samples from probabilities) of hidden layer (replace the zeros below) 
        
        # (student)
        # First we calculate the probability of h given v
        p_h_given_v = sigmoid(self.bias_h + np.dot(visible_minibatch, self.weight_vh))
        # Then we sample from the probabilities to get activations
        h = np.random.binomial(n=1, p=p_h_given_v)

        return p_h_given_v, h


    def get_v_given_h(self,hidden_minibatch):
        
        """Compute probabilities p(v|h) and activations v ~ p(v|h)

        Uses undirected weight "weight_vh" and bias "bias_v"
        
        Args: 
           hidden_minibatch: shape is (size of mini-batch, size of hidden layer)
        Returns:        
           tuple ( p(v|h) , v) 
           both are shaped (size of mini-batch, size of visible layer)
        """
        
        assert self.weight_vh is not None

        n_samples = hidden_minibatch.shape[0]

        if self.is_top:

            """
            Here visible layer has both data and labels. Compute total input for each unit (identical for both cases), \ 
            and split into two parts, something like support[:, :-self.n_labels] and support[:, -self.n_labels:]. \
            Then, for both parts, use the appropriate activation function to get probabilities and a sampling method \
            to get activities. The probabilities as well as activities can then be concatenated back into a normal visible layer.
            """

            # [TODO TASK 4.1] compute probabilities and activations (samples from probabilities) of visible layer (replace the pass below). \
            # Note that this section can also be postponed until TASK 4.2, since in this task, stand-alone RBMs do not contain labels in visible layer.
        
            # (student)
            # First we compute total input for both data and label units
            support = self.bias_v + np.dot(hidden_minibatch, self.weight_vh.T)
            # Then we split the support into data and labels
            support_data = support[:, :-self.n_labels]
            support_label = support[:, -self.n_labels:]
            # Followed by the calculation of v given h for both collections
            p_v_given_h_data = sigmoid(support_data)
            p_v_given_h_label = np.exp(support_label) / np.sum(np.exp(support_label), axis=1, keepdims=True)
            # We can then use the random binomial to get the activations for data (as in get_h_given_v)
            v_data = np.random.binomial(n=1, p=p_v_given_h_data)
            # And use the multinomial distribution to get activations for labels
            v_label = np.random.multinomial(n=1, pvals=p_v_given_h_label)
            # v_label = np.array([
            #     np.random.multinomial(1, pvals)
            #     for pvals in p_v_given_h_label
            # ])
            # Finally we contcatenate data and label probabilities and activations back into
            # a normal visible layer
            p_v_given_h = np.concatenate((p_v_given_h_data, p_v_given_h_label), axis=1)
            v = np.concatenate((v_data, v_label), axis=1)
            
        else:
                        
            # [TODO TASK 4.1] compute probabilities and activations (samples from probabilities) of visible layer (replace the pass and zeros below)             

            # (student)
            # First we calculate the probability of v given h
            p_v_given_h = sigmoid(self.bias_v + np.dot(hidden_minibatch, self.weight_vh.T))
            # Then we sample from the probabilities to get activations (similar to get_h_given_v)
            v = np.random.binomial(n=1, p=p_v_given_h)
        
        return p_v_given_h, v


    
    """ rbm as a belief layer : the functions below do not have to be changed until running a deep belief net """

    

    def untwine_weights(self):
        print(f"UNTWINING WEIGHTS: self.weights_vh: {self.weight_vh.shape}")
        self.weight_v_to_h = np.copy( self.weight_vh )
        print(f"UNTWINING WEIGHTS: self.weights_v_to_h: {self.weight_v_to_h.shape}")
        self.weight_h_to_v = np.copy( np.transpose(self.weight_vh) )
        print(f"UNTWINING WEIGHTS: self.weights_h_to_v: {self.weight_h_to_v.shape}")
        self.weight_vh = None
        print("Weights untwined. weight_vh set to None.")

    def get_h_given_v_dir(self,visible_minibatch):

        """Compute probabilities p(h|v) and activations h ~ p(h|v)

        Uses directed weight "weight_v_to_h" and bias "bias_h"
        
        Args: 
           visible_minibatch: shape is (size of mini-batch, size of visible layer)
        Returns:        
           tuple ( p(h|v) , h) 
           both are shaped (size of mini-batch, size of hidden layer)
        """
        
        assert self.weight_v_to_h is not None

        n_samples = visible_minibatch.shape[0]

        # [TODO TASK 4.2] perform same computation as the function 'get_h_given_v' but with directed connections (replace the zeros below) 
        
        # (student)
        # First we calculate the probability of h given v
        p_h_given_v_direct = sigmoid(self.bias_h + np.dot(visible_minibatch, self.weight_v_to_h))
        # Then we sample from the probabilities to get activations
        h_direct = np.random.binomial(n=1, p=p_h_given_v_direct)

        return p_h_given_v_direct, h_direct


    def get_v_given_h_dir(self,hidden_minibatch):


        """Compute probabilities p(v|h) and activations v ~ p(v|h)

        Uses directed weight "weight_h_to_v" and bias "bias_v"
        
        Args: 
           hidden_minibatch: shape is (size of mini-batch, size of hidden layer)
        Returns:        
           tuple ( p(v|h) , v) 
           both are shaped (size of mini-batch, size of visible layer)
        """
        
        assert self.weight_h_to_v is not None
        
        n_samples = hidden_minibatch.shape[0]
        
        if self.is_top:

            """
            Here visible layer has both data and labels. Compute total input for each unit (identical for both cases), \ 
            and split into two parts, something like support[:, :-self.n_labels] and support[:, -self.n_labels:]. \
            Then, for both parts, use the appropriate activation function to get probabilities and a sampling method \
            to get activities. The probabilities as well as activities can then be concatenated back into a normal visible layer.
            """
            
            # [TODO TASK 4.2] Note that even though this function performs same computation as 'get_v_given_h' but with directed connections,
            # this case should never be executed : when the RBM is a part of a DBN and is at the top, it will have not have directed connections.
            # Appropriate code here is to raise an error (replace pass below)
            
            raise NotImplementedError("This function should not be executed for the top RBM in a DBN, as it does not have directed connections.")
            
            
        else:
                        
            # [TODO TASK 4.2] performs same computaton as the function 'get_v_given_h' but with directed connections (replace the pass and zeros below)             
            # (student)
            # First we calculate the probability of v given h
            p_v_given_h_direct = sigmoid(self.bias_v + np.dot(hidden_minibatch, self.weight_h_to_v.T))
            # Then we sample from the probabilities to get activations (similar to get_h_given_v)
            v_direct = np.random.binomial(n=1, p=p_v_given_h_direct)
        
        return p_v_given_h_direct, v_direct    
        
    def update_generate_params(self,inps,trgs,preds):
        
        """Update generative weight "weight_h_to_v" and bias "bias_v"
        
        Args:
           inps: activities or probabilities of input unit
           trgs: activities or probabilities of output unit (target)
           preds: activities or probabilities of output unit (prediction)
           all args have shape (size of mini-batch, size of respective layer)
        """

        # [TODO TASK 4.3] find the gradients from the arguments (replace the 0s below) and update the weight and bias parameters.
        
        self.delta_weight_h_to_v += 0
        self.delta_bias_v += 0
        
        self.weight_h_to_v += self.delta_weight_h_to_v
        self.bias_v += self.delta_bias_v 
        
        return
    
    def update_recognize_params(self,inps,trgs,preds):
        
        """Update recognition weight "weight_v_to_h" and bias "bias_h"
        
        Args:
           inps: activities or probabilities of input unit
           trgs: activities or probabilities of output unit (target)
           preds: activities or probabilities of output unit (prediction)
           all args have shape (size of mini-batch, size of respective layer)
        """

        # [TODO TASK 4.3] find the gradients from the arguments (replace the 0s below) and update the weight and bias parameters.

        self.delta_weight_v_to_h += 0
        self.delta_bias_h += 0

        self.weight_v_to_h += self.delta_weight_v_to_h
        self.bias_h += self.delta_bias_h
        
        return    
